# Spectrum Objects

`specreduce` has no data class of its own: every operation consumes and returns a `specutils`
**`Spectrum`**, and the `parse_image` of [Chapter 2](02-spectral-images.ipynb) turns whatever you
hand in into one. This chapter covers what that container holds, the last piece of groundwork
before the reduction starts. A `Spectrum` bundles five things:

| part | what it is | why it matters |
|---|---|---|
| `flux` | the data, as an `astropy` `Quantity` | carries its unit, so `electron` and `DN` cannot be confused |
| `spectral_axis` | one coordinate per sample | pixels before calibration, wavelength after |
| `uncertainty` | per-sample variance or standard deviation | drives extraction weights and detection thresholds, and is propagated into every product |
| `mask` | boolean, `True` where data is bad | keeps dead pixels out of fits |
| `wcs` | the pixel-to-wavelength mapping as a transform | how a wavelength solution is stored and travels between programs |

The first four are arrays. The `wcs` holds the same information as the spectral axis, as a
transform rather than a table of values, and has its own section at the end.

In [ ]:
import sys

sys.path.append("../src")

import astropy.units as u
import numpy as np
from astropy.nddata import (
    InverseVariance,
    StdDevUncertainty,
    VarianceUncertainty,
)
from specreduce.core import parse_image
from specutils import Spectrum

from common import read_data

arcs, lamps, obj = read_data()

## Building a `Spectrum` by hand

We start with the constructor, which takes the four pieces directly. Nothing here is specific to
`specreduce`. The five samples stand in for a short stretch of extracted spectrum in electrons,
and their uncertainty follows the CCD noise model.

In [ ]:
signal = np.array([12.0, 40.0, 95.0, 41.0, 11.0])
read_noise = 9.0

demo = Spectrum(
    flux=signal * u.electron,
    spectral_axis=np.arange(signal.size) * u.pix,
    uncertainty=StdDevUncertainty(np.sqrt(signal + read_noise**2)),
    mask=np.array([False, False, False, True, False]),
)

print(f"flux           {demo.flux}")
print(f"spectral_axis  {demo.spectral_axis}")
print(f"uncertainty    {demo.uncertainty.uncertainty_type}, {demo.uncertainty.array}")
print(f"mask           {demo.mask}")
print(f"shape          {demo.shape}")

The uncertainty is stored as a **standard deviation**, and `uncertainty_type` records the
convention; a variance or an inverse variance would be accepted as well.

## A `Spectrum` is not necessarily one-dimensional

Despite its name, `Spectrum` holds anything with a spectral axis, including a whole 2D frame.
`parse_image`, which every `specreduce` operation calls on its input, returns a **2D** `Spectrum`:

In [ ]:
frame = parse_image(obj)

print(f"flux shape           {frame.flux.shape}")
print(f"spectral_axis shape  {frame.spectral_axis.shape}")
print(f"spectral_axis_index  {frame.spectral_axis_index}")
print(f"uncertainty shape    {frame.uncertainty.array.shape}")

`spectral_axis_index` says which array axis the spectral axis describes: 1 here, matching Chapter
2's convention of dispersion on axis 1. The spectral axis has 1000 entries, one per dispersion
column, shared by all 372 rows. The same class therefore covers the 2D frame going in and the 1D
spectrum coming out, and where `specreduce` signatures say "image" or "spectrum", both are
`Spectrum` objects of different dimensionality.

## Uncertainties: three conventions, one quantity

`astropy.nddata` expresses a per-sample uncertainty as `StdDevUncertainty`, `VarianceUncertainty`
or `InverseVariance`, three forms of the same information, and `represent_as` converts between
them, unit included:

In [ ]:
sigma = StdDevUncertainty([2.0, 3.0, 5.0], unit=u.electron)

for uncertainty in (
    sigma,
    sigma.represent_as(VarianceUncertainty),
    sigma.represent_as(InverseVariance),
):
    print(
        f"{uncertainty.uncertainty_type:>4}  {uncertainty.array}  [{uncertainty.unit}]"
    )

`specreduce` accepts any of the three. Every operation converts its input to the form it computes
in, on a copy, and returns its result in the convention it received, so the class is a choice of
representation, not a constraint. The book's frames carry a `StdDevUncertainty`, as the data files
store it, and every product comes back the same way. `represent_as` is for code of your own, for
example to add independent uncertainties in quadrature as variances.

## Units

A `Spectrum` carries two independent units, one on the flux and one on the spectral axis. The
**spectral axis** converts freely, since wavelength, frequency and energy describe the same
physical quantity:

In [ ]:
calibrated_axis = Spectrum(
    flux=signal * u.electron,
    spectral_axis=(5000 + np.arange(signal.size) * 5.0) * u.AA,
    uncertainty=StdDevUncertainty(np.sqrt(signal + read_noise**2)),
)

print(f"wavelength  {calibrated_axis.spectral_axis}")
print(f"frequency   {calibrated_axis.spectral_axis.to(u.THz).round(3)}")
print(
    f"energy      {calibrated_axis.spectral_axis.to(u.eV, equivalencies=u.spectral()).round(4)}"
)

The **flux** unit does not, and that is a feature:

In [ ]:
try:
    calibrated_axis.with_flux_unit(u.ct)
except u.UnitConversionError as error:
    print(f"UnitConversionError: {error}")

scaled = calibrated_axis.with_flux_unit(u.Unit(1e3 * u.electron))
print(f"\nrescaling the same unit is fine: {scaled.flux}")

`electron` and `ct` (counts, ADU) are not convertible without the detector gain, and `astropy`
refuses to guess. The frames were converted to electrons with `GAIN` before the book starts
(Chapter 1), because photon statistics apply to detected charge rather than to digitized counts,
and a container that relabeled one as the other would erase that distinction.

## The spectral axis and the WCS

The `spectral_axis` is a `SpectralAxis`, a `Quantity` subclass with one entry per sample: a pixel
index before wavelength calibration, a wavelength (or frequency, or energy) after. On a 2D frame
there is still one axis, shared by every row. The `wcs` holds the same information as a transform
from array index to world coordinate that can be evaluated anywhere, not only at the sample
centers. A `Spectrum` is built from either: given a `spectral_axis`, `specutils` builds a
lookup-table GWCS from its values, and given a `wcs`, it evaluates the transform at every pixel to
fill in the axis. Both are always populated, and `spectral_axis` is the attribute to read in
ordinary code. On a pixel axis the world coordinate has no physical meaning, and `astropy` will
not build a `SpectralCoord` from it, so before calibration only `spectral_axis` is usable.

The `wcs` matters when a spectrum with a wavelength solution is written out and read back, and it
comes in two standards that `astropy`, `specutils` and `specreduce` all support. The **FITS
spectral WCS** of [Greisen et al. (2006)](https://ui.adsabs.harvard.edu/abs/2006A%26A...446..747G/abstract) lives in header keywords, offers a fixed set of dispersion functions selected
by `CTYPE` (linear, logarithmic, the grism dispersion relation, a lookup table), and is read by
every FITS-aware program. It is the portable choice, but a fitted solution has to be approximated
by one of its functions. **GWCS**, the Generalized World Coordinate System developed for JWST,
represents the mapping as a chain of `astropy.modeling` models, stores any fitted polynomial
exactly, and is serialized to ASDF rather than to a FITS header. The wavelength solutions of
`specreduce` expose their exact model as a GWCS and can fit the FITS grism function to it for
export, as Chapter 10 shows.

## Summary

- Every `specreduce` operation consumes and returns a `specutils` **`Spectrum`**: `flux` (a
  `Quantity`), `spectral_axis`, `uncertainty`, `mask` and `wcs` in one object.
- **A `Spectrum` can be 2D.** `parse_image` returns the whole frame as one, with
  `spectral_axis_index` saying which axis is spectral (1, per Chapter 2's convention).
- Uncertainties come in three interchangeable conventions that `represent_as` converts between.
  `specreduce` accepts all three and returns the one it was given; the book's data carry a
  `StdDevUncertainty`.
- The spectral axis converts freely between wavelength, frequency and energy. The flux unit does
  not: `electron` and `ct` need the gain, and `astropy` will not guess.
- `spectral_axis` gives one coordinate per sample and `wcs` the same mapping as a transform.
  `specutils` derives either from the other, and `spectral_axis` is the one to read in ordinary
  code.
- The `wcs` comes in two standards, the FITS spectral WCS (portable, a fixed set of dispersion
  functions) and GWCS (exact, any model chain, serialized to ASDF), and `specreduce` supports both.

That completes the foundations. Next: [Tracing the Spectrum](04-tracing.ipynb), where the reduction
proper begins.